In [58]:
import pandas as pd
import os

In [59]:
DEST_FILE = "../data"
FILE_NAME = "TMDB_all_movies.csv"
full_path = os.path.join(DEST_FILE, FILE_NAME)

In [60]:
# Lecture du CSV
df = pd.read_csv(full_path)

In [61]:
# Tri
df_sorted = df.sort_values(
    by=["vote_count", "popularity", "vote_average"],
    ascending=[False, False, False]
)

In [62]:
# Garder seulement les 50000 premiers
df = df_sorted.head(50000).copy()

In [63]:
# Conversions numériques
df["vote_average"] = pd.to_numeric(df["vote_average"], errors="coerce").astype(float)
df["vote_count"] = pd.to_numeric(df["vote_count"], errors="coerce").astype("Int64")  # Int64 pour accepter NaN
df["runtime"] = pd.to_numeric(df["runtime"], errors="coerce").astype(float)
df["budget"] = pd.to_numeric(df["budget"], errors="coerce").astype(float)
df["popularity"] = pd.to_numeric(df["popularity"], errors="coerce").astype(float)

In [64]:
# Dates
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year.astype("Int64")

In [65]:
# Colonnes à transformer en listes
array_columns = [
    "genres", "production_countries", "production_companies", "cast", "director", "writers"
]

for col in array_columns:
    df[col + "_array"] = (
        df[col].str.split(r",\s*")       # découper sur virgule + espace
             .apply(lambda x: x if isinstance(x, list) else [])  # remplacer NaN par []
    )

# Suppression des colonnes originales
df = df.drop(columns=array_columns)

In [66]:
# Vérifier les types
print(df.dtypes)

id                                     int64
title                                 object
vote_average                         float64
vote_count                             Int64
status                                object
release_date                  datetime64[ns]
revenue                              float64
runtime                              float64
budget                               float64
imdb_id                               object
original_language                     object
original_title                        object
overview                              object
popularity                           float64
tagline                               object
spoken_languages                      object
director_of_photography               object
producers                             object
music_composer                        object
imdb_rating                          float64
imdb_votes                           float64
poster_path                           object
release_ye

In [67]:
df = df.drop(
    ["status", "imdb_id", "tagline", "director_of_photography",
     "producers", "imdb_rating", "imdb_votes",
     "music_composer", "revenue", "spoken_languages", "original_language"],
    axis=1
)

In [68]:
# Delete line with empty title
df = df[df["title"].notna() & (df["title"] != "")]

In [69]:
# Delete line with empty overview
df = df[df["overview"].notna() & (df["overview"] != "")]

In [70]:
df = df.fillna({
    'release_year': -1
})

In [71]:
#  ne garder que les lignes qui ont un runtime supérieur à 44.0
df = df[df["runtime"] > 44.0]

In [72]:
# enlever les films qui ont uniquement la valeur 'Documentary' dans la colonne genres
df = df[~df["genres_array"].apply(lambda x: len(x) == 1 and x[0] == "Documentary")]

In [73]:
df.shape

(45355, 18)

In [74]:
def count_empty_values(df):
    counts = {}
    for col in df.columns:
        counts[col] = (
            df[col].isna()                                  # NaN / None
            | (df[col] == "")                               # chaîne vide
            | (df[col].apply(lambda x: isinstance(x, list) and len(x) == 0))  # liste vide
        ).sum()
    return pd.Series(counts, name="empty_count")

In [75]:
empty_counts = count_empty_values(df)
print(empty_counts)

id                               0
title                            0
vote_average                     0
vote_count                       0
release_date                     2
runtime                          0
budget                           0
original_title                   0
overview                         0
popularity                       0
poster_path                     78
release_year                     0
genres_array                    31
production_countries_array     568
production_companies_array    1856
cast_array                      50
director_array                  78
writers_array                 1009
Name: empty_count, dtype: int64


In [76]:
clean_path = os.path.join(DEST_FILE, "TMDB_clean.csv")

In [77]:
if os.path.exists(clean_path):
    os.remove(clean_path)

In [78]:
df.to_csv(clean_path, index=False)

## ML

In [79]:
from sklearn.feature_extraction.text import TfidfVectorizer
import umap.umap_ as umap
from sklearn.cluster import KMeans
import numpy as np

In [ ]:
# Vectorisation TF-IDF sur les résumés
# faire des tests de temps en temps avec max_features=1000
vectorizer = TfidfVectorizer(max_features=500, stop_words="english")
X_tfidf = vectorizer.fit_transform(df["overview"])

In [107]:
# Réduction dimensionnelle avec UMAP
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
embedding = reducer.fit_transform(X_tfidf.toarray())

df["x"] = embedding[:,0]
df["y"] = embedding[:,1]

/usr/local/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [108]:
# Clustering KMeans (nombre de clusters à ajuster)
n_clusters = 1
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
df["cluster"] = kmeans.fit_predict(embedding)

In [109]:
def find_closest_movie(df, input_title):
    # Cherche le film dans le dataset
    if input_title not in df['title'].values:
        print(f"Le film '{input_title}' n'a pas été trouvé dans le dataset.")
        return None
    
    # Coordonnées du film input
    input_point = df.loc[df['title'] == input_title, ['x', 'y']].values[0]
    
    # Calcul des distances euclidiennes vers tous les points
    df['distance'] = np.linalg.norm(df[['x', 'y']].values - input_point, axis=1)
    
    # Exclure le film lui-même (distance = 0)
    df_filtered = df[df['title'] != input_title]
    
    # Trouver le film avec la distance minimale
    closest_movie = df_filtered.loc[df_filtered['distance'].idxmin()]
    
    return closest_movie[['title', 'distance', 'cluster']]

In [ ]:
input_title = "The Godfather"  # Exemple de film à rechercher
closest = find_closest_movie(df, input_title)

if closest is not None:
    print(f"Le film le plus proche de '{input_title}' est '{closest['title']}' dans le cluster {closest['cluster']} à une distance de {closest['distance']:.3f}")

Le film le plus proche de 'Schindler's List' est 'Catch-22' dans le cluster 0 à une distance de 0.009
